In [ ]:
import numpy as np
import pandas as pd
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
transform = transforms.Compose([
    transforms.Resize((16, 16)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
trainDataset = datasets.MNIST(root="", train=True, download=True, transform=transform)
trainDataloader = torch.utils.data.DataLoader(trainDataset, batch_size=64, shuffle=True)

In [ ]:
testDataset = datasets.MNIST(root="", train=False, download=True, transform=transform)
testDataloader = torch.utils.data.DataLoader(testDataset, batch_size=64, shuffle=True)

In [ ]:
class CNNModel(nn.Module):
  def __init__(self, outputSize):
    super().__init__()

    self.Model = nn.Sequential(
        nn.Conv2d(1,16,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(16,32,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(32,64,2,1,2),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Flatten(),

        nn.Linear(64*4*4, outputSize)
    )

  def forward(self, x):
    return self.Model(x)

In [ ]:
cnnModel = CNNModel(10).to(device)

In [ ]:
cnnOptimizer = optim.Adam(cnnModel.parameters(), lr=0.001)

In [ ]:
criteria = nn.CrossEntropyLoss()

In [ ]:
epochs = 5

In [ ]:
for epoch in range(epochs):
  totalLoss = 0.0
  count = 0
  for image, label in trainDataloader:

    image = image.to(device)
    label = label.to(device)

    op = cnnModel(image)

    loss = criteria(op, label)
    totalLoss += loss.item()
    count += 1

    cnnOptimizer.zero_grad()
    loss.backward()
    cnnOptimizer.step()

  avgLoss = totalLoss/count
  print(f"Epoch: {epoch+1}, Loss: {avgLoss:.4}")

Epoch: 1, Loss: 0.383
Epoch: 2, Loss: 0.107
Epoch: 3, Loss: 0.0768
Epoch: 4, Loss: 0.06264
Epoch: 5, Loss: 0.05328


In [ ]:
cnnModel.eval()

correct = 0

with torch.no_grad():
  for img,label in testDataloader:

    img = img.to(device)
    label = label.to(device)

    op = cnnModel(img)

    _, prediction = torch.max(op,1)
    correct += (prediction == label).sum().item()

accuracy = correct/len(testDataset)
print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 98.34%
